# Case Bank Manager
Manage the `Soccer_Case_Bank` Qdrant collection for few-shot planning examples.

**Full rewrite flow:** run Cell 1 (setup) → Cell 2 ((re)create: drops + recreates collection with indexes) → Cell 4 (embed + upsert all rows from CSV).

**Cells:**
1. Setup & imports
2. (Re)create collection — drops if exists, then creates + payload indexes
3. Delete collection (standalone utility, interactive)
4. Batch upsert from `planning_fewshot_examples.csv`

In [ ]:
import os
import json
import uuid
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PayloadSchemaType,
    PointStruct, Filter, FieldCondition, MatchValue
)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from tqdm import tqdm

load_dotenv(override=True)

QDRANT_URL = os.getenv('QDRANT_URL')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
COLLECTION_NAME = os.getenv('QDRANT_CASE_BANK_COLLECTION_NAME', 'Soccer_Case_Bank')
CSV_PATH = os.path.join(os.getcwd(), 'planning_fewshot_examples.csv')

# gRPC protocol (consistent with the app's Qdrant clients). prefer_grpc lets the
# client use the gRPC port (6334) instead of REST; pass the URL as-is.
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    prefer_grpc=True,
    check_compatibility=False,
)
embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-001',
    output_dimensionality=768,
    google_api_key=GOOGLE_API_KEY,
    task_type='RETRIEVAL_DOCUMENT'
)
print(f'✅ Connected to Qdrant (gRPC): {QDRANT_URL}')
print(f'✅ Collection: {COLLECTION_NAME}')

In [ ]:
# ── Cell 2: (Re)create collection — DROP if exists, then create + indexes ─────
# Full rewrite: run this to reset the collection before re-seeding from CSV.
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f'🗑️  Dropped existing collection "{COLLECTION_NAME}".')

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)
# Create payload indexes for fast filtered search
client.create_payload_index(COLLECTION_NAME, 'has_media', PayloadSchemaType.BOOL)
client.create_payload_index(COLLECTION_NAME, 'label', PayloadSchemaType.KEYWORD)
client.create_payload_index(COLLECTION_NAME, 'use_case', PayloadSchemaType.KEYWORD)
print(f'✅ Collection "{COLLECTION_NAME}" created with indexes: has_media, label, use_case')

In [2]:
# ── Cell 3: Delete collection ────────────────────────────────────────────────
confirm = input(f'Type DELETE to confirm deletion of "{COLLECTION_NAME}": ')
if confirm.strip() == 'DELETE':
    client.delete_collection(COLLECTION_NAME)
    print(f'🗑️  Collection "{COLLECTION_NAME}" deleted.')
else:
    print('Aborted.')

🗑️  Collection "Soccer_Case_Bank" deleted.


In [ ]:
# ── Cell 4: Batch upsert from CSV ────────────────────────────────────────────
BATCH_SIZE = 32

def _to_bool(v) -> bool:
    # CSV may yield a python bool (pandas) or the strings "true"/"false";
    # bool("false") is True, so parse explicitly.
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() == 'true'

df = pd.read_csv(CSV_PATH)
print(f'📄 Loaded {len(df)} rows from CSV')

rows = df.to_dict('records')
total_upserted = 0

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc='Upserting batches'):
    batch = rows[batch_start: batch_start + BATCH_SIZE]
    texts = [str(r['query']) for r in batch]
    vectors = embeddings.embed_documents(texts)

    points = []
    for row, vector in zip(batch, vectors):
        payload = {
            'query': str(row['query']),
            'has_media': _to_bool(row['has_media']),
            'label': str(row['label']),
            'use_case': str(row['use_case']),
            'tool_chains': str(row['tool_chains']),
            'sub_queries': str(row['sub_queries']),
            'reasoning': str(row['reasoning']),
        }
        points.append(PointStruct(id=str(uuid.uuid4()), vector=vector, payload=payload))

    client.upsert(collection_name=COLLECTION_NAME, points=points)
    total_upserted += len(points)

print(f'✅ Upserted {total_upserted} points into "{COLLECTION_NAME}"')
info = client.get_collection(COLLECTION_NAME)
print(f'📊 Collection points count: {info.points_count}')

In [4]:
info.points_count

84